# Pipeline de Preprocesado — Proyecto Sushi

Aplicación de técnicas clásicas de visión artificial sobre imágenes de sushi.

**Técnicas cubiertas:**
1. Espacios de color (Grayscale, HSV, LAB) e histogramas
2. Detección de bordes (Canny, Sobel, Laplacian)
3. Umbralización (Binary, Otsu, Adaptativa) y ecualización
4. Análisis de contornos y características geométricas
5. Operaciones morfológicas
6. Segmentación (Watershed y K-means)
7. Descriptores de características (SIFT, ORB, HOG, Gabor)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Imagen de muestra — nigiri como representante del dataset
SAMPLES_DIR = Path('../data/samples')
img_path = SAMPLES_DIR / 'sushi-nigiri.jpg'

bgr  = cv2.imread(str(img_path))
rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(4, 4))
plt.imshow(rgb)
plt.title(f'Imagen original: {img_path.name}  ({bgr.shape[1]}x{bgr.shape[0]})')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Forma: {bgr.shape}  |  dtype: {bgr.dtype}')

## 1. Espacios de color e histogramas

In [ ]:
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)

h_ch, s_ch, v_ch = cv2.split(hsv)
l_ch, a_ch, b_ch = cv2.split(lab)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Espacios de color', fontsize=14)

pairs = [
    (axes[0,0], rgb,   'RGB original',     None),
    (axes[0,1], gray,  'Escala de grises',  'gray'),
    (axes[0,2], h_ch,  'HSV — Canal H',     'hsv'),
    (axes[0,3], s_ch,  'HSV — Canal S',     'gray'),
    (axes[1,0], v_ch,  'HSV — Canal V',     'gray'),
    (axes[1,1], l_ch,  'LAB — Canal L',     'gray'),
    (axes[1,2], a_ch,  'LAB — Canal a',     'RdYlGn'),
    (axes[1,3], b_ch,  'LAB — Canal b',     'coolwarm'),
]
for ax, img, title, cmap in pairs:
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Histogramas de intensidad', fontsize=13)

axes[0].hist(gray.ravel(), bins=256, range=[0,256], color='gray', edgecolor='none')
axes[0].set_title('Escala de grises')
axes[0].set_xlabel('Intensidad'); axes[0].set_ylabel('Frecuencia')

axes[1].hist(v_ch.ravel(), bins=256, range=[0,256], color='darkorange', edgecolor='none')
axes[1].set_title('Canal V (HSV — brillo)')
axes[1].set_xlabel('Valor')

for ch, col, lbl in zip(cv2.split(bgr), ('blue','green','red'), ('B','G','R')):
    axes[2].hist(ch.ravel(), bins=256, range=[0,256], alpha=0.5, color=col, label=lbl, edgecolor='none')
axes[2].set_title('Canales BGR')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Detección de bordes

In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Canny con tres niveles de umbral
canny_low    = cv2.Canny(blur, 30, 100)
canny_medium = cv2.Canny(blur, 50, 150)
canny_high   = cv2.Canny(blur, 100, 200)

# Sobel
sobel_x  = cv2.convertScaleAbs(cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3))
sobel_y  = cv2.convertScaleAbs(cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3))
sobel_xy = cv2.convertScaleAbs(cv2.magnitude(
    cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3),
    cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
))

# Laplacian
laplacian = cv2.convertScaleAbs(cv2.Laplacian(blur, cv2.CV_64F))

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Detección de bordes', fontsize=14)

pairs = [
    (axes[0,0], gray,         'Original (gris)'),
    (axes[0,1], canny_low,    'Canny bajo (30–100)'),
    (axes[0,2], canny_medium, 'Canny medio (50–150)'),
    (axes[0,3], canny_high,   'Canny alto (100–200)'),
    (axes[1,0], blur,         'Gaussian blur σ=0'),
    (axes[1,1], sobel_x,      'Sobel X'),
    (axes[1,2], sobel_y,      'Sobel Y'),
    (axes[1,3], sobel_xy,     'Sobel combinado'),
]
for ax, img, title in pairs:
    ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Laplacian max={laplacian.max():.0f}  —  más alto = más bordes nítidos')

## 3. Umbralización y transformaciones geométricas

In [ ]:
_, thresh_bin      = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
_, thresh_inv      = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
otsu_val, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
thresh_adapt_mean  = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                           cv2.THRESH_BINARY, 11, 2)
thresh_adapt_gauss = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                           cv2.THRESH_BINARY, 11, 2)
equalized = cv2.equalizeHist(gray)

h, w = gray.shape
M_rot = cv2.getRotationMatrix2D((w//2, h//2), 45, 1.0)
rotated = cv2.warpAffine(bgr, M_rot, (w, h))

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Umbralización', fontsize=14)

pairs = [
    (axes[0,0], gray,               'Original (gris)',                 'gray'),
    (axes[0,1], thresh_bin,         'Binario (127)',                   'gray'),
    (axes[0,2], thresh_inv,         'Invertido',                       'gray'),
    (axes[0,3], thresh_otsu,        f'Otsu (umbral={otsu_val:.0f})',   'gray'),
    (axes[1,0], thresh_adapt_mean,  'Adaptativo — media',              'gray'),
    (axes[1,1], thresh_adapt_gauss, 'Adaptativo — Gaussiano',          'gray'),
    (axes[1,2], equalized,          'Ecualización de histograma',      'gray'),
    (axes[1,3], cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB), 'Rotación 45°', None),
]
for ax, img, title, cmap in pairs:
    ax.imshow(img, cmap=cmap); ax.set_title(title, fontsize=9); ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Umbral Otsu calculado automáticamente: {otsu_val:.0f}')

## 4. Análisis de contornos y características geométricas

In [ ]:
MIN_AREA = 500

blur2    = cv2.GaussianBlur(gray, (5,5), 0)
_, binary = cv2.threshold(blur2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]

result = bgr.copy()
cv2.drawContours(result, contours, -1, (0, 255, 0), 2)

features = []
for i, cnt in enumerate(contours):
    area       = cv2.contourArea(cnt)
    perimeter  = cv2.arcLength(cnt, True)
    circularity = (4 * np.pi * area / perimeter**2) if perimeter > 0 else 0
    x, y, bw, bh = cv2.boundingRect(cnt)
    aspect_ratio = bw / bh if bh > 0 else 0
    M = cv2.moments(cnt)
    cx = int(M['m10']/M['m00']) if M['m00'] > 0 else x + bw//2
    cy = int(M['m01']/M['m00']) if M['m00'] > 0 else y + bh//2
    features.append({'id': i+1, 'area': area, 'circularity': circularity,
                     'aspect_ratio': aspect_ratio, 'center': (cx, cy)})
    cv2.circle(result, (cx, cy), 5, (0,0,255), -1)
    cv2.putText(result, f'#{i+1} C:{circularity:.2f}',
                (x, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,0), 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Análisis de contornos — {len(contours)} detectados (área ≥ {MIN_AREA})', fontsize=13)
axes[0].imshow(rgb);                                    axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(binary, cmap='gray');                    axes[1].set_title('Umbral Otsu'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)); axes[2].set_title('Contornos anotados'); axes[2].axis('off')
plt.tight_layout()
plt.show()

# Tabla de características
print(f"{'#':>3}  {'Área':>8}  {'Circularidad':>13}  {'Aspecto':>8}  Centro")
print('-' * 55)
for f in features:
    print(f"{f['id']:>3}  {f['area']:>8.0f}  {f['circularity']:>13.3f}  {f['aspect_ratio']:>8.2f}  {f['center']}")

## 5. Operaciones morfológicas

In [ ]:
_, binary_m = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

k_sq  = cv2.getStructuringElement(cv2.MORPH_RECT,    (5,5))
k_ell = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

eroded   = cv2.erode(binary_m, k_sq)
dilated  = cv2.dilate(binary_m, k_sq)
opened   = cv2.morphologyEx(binary_m, cv2.MORPH_OPEN,     k_ell)
closed   = cv2.morphologyEx(binary_m, cv2.MORPH_CLOSE,    k_ell)
gradient = cv2.morphologyEx(binary_m, cv2.MORPH_GRADIENT, k_sq)
tophat   = cv2.morphologyEx(binary_m, cv2.MORPH_TOPHAT,   k_sq)
blackhat = cv2.morphologyEx(binary_m, cv2.MORPH_BLACKHAT, k_sq)
cleaned  = cv2.morphologyEx(cv2.morphologyEx(binary_m, cv2.MORPH_CLOSE, k_ell, iterations=2),
                            cv2.MORPH_OPEN, k_ell)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Operaciones morfológicas', fontsize=14)

pairs = [
    (axes[0,0], binary_m, 'Otsu binario'),
    (axes[0,1], eroded,   'Erosión (rect 5×5)'),
    (axes[0,2], dilated,  'Dilatación (rect 5×5)'),
    (axes[0,3], opened,   'Apertura (elipse 5×5)'),
    (axes[1,0], closed,   'Cierre (elipse 5×5)'),
    (axes[1,1], gradient, 'Gradiente morfológico'),
    (axes[1,2], tophat,   'Top-hat'),
    (axes[1,3], cleaned,  'Limpieza (cierre + apertura)'),
]
for ax, img, title in pairs:
    ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Segmentación — Watershed y K-means

In [ ]:
# ── Watershed ─────────────────────────────────────────────────────────────────
def watershed_seg(gray, bgr):
    blur_w  = cv2.GaussianBlur(gray, (5,5), 0)
    _, th   = cv2.threshold(blur_w, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    opening = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=2)
    bg      = cv2.dilate(opening, kernel, iterations=3)
    dist    = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, fg   = cv2.threshold(dist, 0.5*dist.max(), 255, 0)
    fg      = fg.astype(np.uint8)
    unknown = cv2.subtract(bg, fg)
    _, markers = cv2.connectedComponents(fg)
    markers += 1
    markers[unknown == 255] = 0
    result = bgr.copy()
    cv2.watershed(result, markers)
    result[markers == -1] = [0, 0, 255]
    return result

# ── K-means ───────────────────────────────────────────────────────────────────
def kmeans_seg(bgr, k=3):
    pixels   = bgr.reshape((-1,3)).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    return centers.astype(np.uint8)[labels.flatten()].reshape(bgr.shape)

ws_result     = watershed_seg(gray, bgr)
km_result_k3  = kmeans_seg(bgr, k=3)
km_result_k5  = kmeans_seg(bgr, k=5)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Segmentación', fontsize=14)
axes[0].imshow(rgb);                                          axes[0].set_title('Original');      axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(ws_result, cv2.COLOR_BGR2RGB));   axes[1].set_title('Watershed');     axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(km_result_k3, cv2.COLOR_BGR2RGB)); axes[2].set_title('K-means k=3'); axes[2].axis('off')
axes[3].imshow(cv2.cvtColor(km_result_k5, cv2.COLOR_BGR2RGB)); axes[3].set_title('K-means k=5'); axes[3].axis('off')
plt.tight_layout()
plt.show()

## 7. Descriptores de características — SIFT, ORB, HOG, Gabor

In [ ]:
# ── SIFT ──────────────────────────────────────────────────────────────────────
sift = cv2.SIFT_create(nfeatures=200)
kp_sift, _ = sift.detectAndCompute(gray, None)
sift_img = cv2.drawKeypoints(bgr, kp_sift, None,
                              flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# ── ORB ───────────────────────────────────────────────────────────────────────
orb = cv2.ORB_create(nfeatures=200)
kp_orb, _ = orb.detectAndCompute(gray, None)
orb_img = cv2.drawKeypoints(bgr, kp_orb, None,
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# ── HOG ───────────────────────────────────────────────────────────────────────
try:
    from skimage.feature import hog
    from skimage import exposure
    resized   = cv2.resize(gray, (128, 128))
    _, hog_img = hog(resized, orientations=8, pixels_per_cell=(16,16),
                     cells_per_block=(1,1), visualize=True)
    hog_img = (exposure.rescale_intensity(hog_img, in_range=(0,10)) * 255).astype(np.uint8)
except ImportError:
    hog_img = gray.copy()
    print('scikit-image no instalado — pip install scikit-image')

# ── Gabor ─────────────────────────────────────────────────────────────────────
def gabor_bank(g):
    combined = np.zeros_like(g, dtype=np.float32)
    for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
        for sigma in [1, 3]:
            k = cv2.getGaborKernel((21,21), sigma, theta, 10.0, 0.5, 0, ktype=cv2.CV_32F)
            f = cv2.filter2D(g.astype(np.float32), -1, k)
            combined = np.maximum(combined, np.abs(f))
    return cv2.normalize(combined, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

gabor_out = gabor_bank(gray)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle(f'Descriptores de características  |  SIFT: {len(kp_sift)} kp  ORB: {len(kp_orb)} kp', fontsize=13)
axes[0].imshow(cv2.cvtColor(sift_img, cv2.COLOR_BGR2RGB)); axes[0].set_title(f'SIFT ({len(kp_sift)} keypoints)'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(orb_img,  cv2.COLOR_BGR2RGB)); axes[1].set_title(f'ORB ({len(kp_orb)} keypoints)');  axes[1].axis('off')
axes[2].imshow(hog_img, cmap='gray');                       axes[2].set_title('HOG (8 orientaciones)');           axes[2].axis('off')
axes[3].imshow(gabor_out, cmap='gray');                     axes[3].set_title('Banco Gabor (4 ángulos × 2 σ)');   axes[3].axis('off')
plt.tight_layout()
plt.show()

## Resumen

| Técnica | Resultado observado en sushi |
|---------|-----------------------------|
| Espacios de color | Canal V (HSV) separa bien el brillo del fondo; canal a (LAB) distingue tonos rojizos del salmón |
| Canny medio | Detecta correctamente el contorno exterior del nigiri |
| Otsu | Umbral calculado automáticamente separa el sushi del fondo |
| Contornos | Circularity ≈ 0.7–0.9 confirma la forma redondeada característica del nigiri |
| Morfología | Apertura elimina ruido; cierre rellena huecos internos |
| Watershed | Segmenta regiones individuales por marcadores de distancia |
| K-means k=3 | Agrupa píxeles en fondo / arroz / topping |
| SIFT/ORB | Concentra keypoints en zonas texturizadas (grano de arroz, borde del topping) |
| Gabor | Responde a texturas periódicas del arroz |